In [1]:
import pandas as pd
import pandera as pa
from pandera import DataFrameSchema, Column, Check
from pandera.dtypes import Float, String, Int, Bool
from pathlib import Path
import warnings
from datetime import datetime, timezone
from pydantic import BaseModel, Field, field_validator
warnings.filterwarnings("ignore")

df = pd.read_csv("dataset1.csv")
df.drop(columns=["Unnamed: 0"], inplace=True)
UNIQUE_TRACK_GENRES = list(df["track_genre"].unique())
N_ROWS_TO_TEST = [1, 10, 100, 1000, 2500]

def strict_datetime_column(series: pd.Series) -> bool:
    def is_strict_format(val):
        try:
            datetime.strptime(val, "%Y-%m-%d %H:%M:%S UTC")
            return True
        except:
            return False

    mask = series.apply(is_strict_format)
    return mask


music_schema = pa.DataFrameSchema(
    columns={
        "track_id": Column(
            String,
            checks=Check.str_length(22)
        ),
        "artists": Column(
            String,
            checks=Check.str_length(2, 512)
        ),
        "album_name": Column(
            String,
            checks=Check.str_length(2, 512)
        ),
        "track_name": Column(
            String,
            checks=Check.str_length(2, 512)
        ),
        "popularity": Column(
            Int,
            checks=Check.in_range(0, 100)
        ),
        "duration_ms": Column(
            Int,
            checks=Check.in_range(1, 5237760)
        ),
        "explicit": Column(
            Bool
        ),
        "danceability": Column(
            Float,
            checks=Check.in_range(0.0, 1.0)
        ),
        "energy": Column(
            Float,
            checks=Check.in_range(0.0, 1.0)
        ),
        "key": Column(
            Int,
            checks=Check.in_range(0, 11)
        ),
        "loudness": Column(
            Float,
            checks=Check.in_range(-45, 5)
        ),
        "mode": Column(
            Float,
            checks=Check.in_range(0.0, 1.0)
        ),
        "speechiness": Column(
            Float,
            checks=Check.in_range(0.0, 1.0)
        ),
        "acousticness": Column(
            Float,
            checks=Check.in_range(0.0, 1.0)
        ),
        "instrumentalness": Column(
            Float,
            checks=Check.in_range(0.0, 1.0)
        ),
        "liveness": Column(
            Float,
            checks=Check.in_range(0.0, 1.0)
        ),
        "valence": Column(
            Float,
            checks=Check.in_range(0.0, 1.0)
        ),
        "tempo": Column(
            Float,
            checks=Check.in_range(0.0, 256.0)
        ),
        "time_signature": Column(
            Int,
            checks=Check.in_range(0, 5)
        ),
        "track_genre": Column(
            String,
            checks=[
                Check(lambda s: s.nunique() <= 114, error="Wrong number of unique track genres"),
                Check.isin(UNIQUE_TRACK_GENRES, error="Invalid track genre")
            ]
        ),
    },
    strict=True,
    coerce=True,
    ordered=False
)

In [2]:
STUDENT_NAME = "Calestru"
DATETIME = datetime.now().strftime("%Y%m%d")

try:
    music_schema.validate(df, lazy=True)
except pa.errors.SchemaErrors as err:
    errors_df = err.failure_cases
    errors_df["failure_case"] = errors_df["failure_case"].astype(str)
    errors_df.sort_values(by='index', inplace=True)
    errors_df.reset_index(drop=True, inplace=True)
    errors_df.to_parquet(f"{STUDENT_NAME}_{DATETIME}_validation_report.parquet", index=False) 

errors_df

,schema_context,column,check,check_number,failure_case,index
0,Column,album_name,"str_length(2, 512)",0,0,650
1,Column,album_name,"str_length(2, 512)",0,E,910
2,Column,album_name,"str_length(2, 512)",0,V,1456
3,Column,album_name,"str_length(2, 512)",0,4,1629
4,Column,album_name,"str_length(2, 512)",0,&,3911
...,...,...,...,...,...,...
233,Column,album_name,"str_length(2, 512)",0,5,111950
234,Column,album_name,"str_length(2, 512)",0,9,112209
235,Column,album_name,"str_length(2, 512)",0,O,112475
236,Column,album_name,"str_length(2, 512)",0,X,112815


In [3]:
import pandas as pd

for nrows in N_ROWS_TO_TEST:
    df_timing = pd.read_csv("dataset1.csv", nrows=nrows)
    df_timing.drop(columns=["Unnamed: 0"], inplace=True)

    df_timing = df_timing.dropna()
    text_cols = ["artists", "album_name", "track_name"]
    df_timing = df_timing[df_timing[text_cols].apply(lambda s: s.str.len().between(2, 512)).all(axis=1)]

    %timeit music_schema.validate(df_timing)

29.8 ms ± 6.54 ms per loop (mean ± std. dev. of 7 runs, 100 loops each)
23.3 ms ± 3.61 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)
27.3 ms ± 5.36 ms per loop (mean ± std. dev. of 7 runs, 100 loops each)
24.2 ms ± 4.98 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)
33.9 ms ± 7.86 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [4]:
class MusicTrack(BaseModel):
    track_id: str = Field(..., min_length=22, max_length=22)
    artists: str = Field(..., min_length=2, max_length=512)
    album_name: str = Field(..., min_length=2, max_length=512)
    track_name: str = Field(..., min_length=2, max_length=512)
    popularity: int = Field(..., ge=0, le=100)
    duration_ms: int = Field(..., ge=1, le=5237760)
    explicit: bool
    danceability: float = Field(..., ge=0.0, le=1.0)
    energy: float = Field(..., ge=0.0, le=1.0)
    key: int = Field(..., ge=0, le=11)
    loudness: float = Field(..., ge=-45, le=5)
    mode: float = Field(..., ge=0.0, le=1.0)
    speechiness: float = Field(..., ge=0.0, le=1.0)
    acousticness: float = Field(..., ge=0.0, le=1.0)
    instrumentalness: float = Field(..., ge=0.0, le=1.0)
    liveness: float = Field(..., ge=0.0, le=1.0)
    valence: float = Field(..., ge=0.0, le=1.0)
    tempo: float = Field(..., ge=0.0, le=256.0)
    time_signature: int = Field(..., ge=0, le=5)
    track_genre: str

    @field_validator("track_genre")
    def validate_track_genre(cls, v):
        if v not in UNIQUE_TRACK_GENRES:
            raise ValueError("Invalid track genre")
        return v

    class Config:
        extra = "forbid"

In [5]:
def validation_pipe(df: pd.DataFrame):
    for idx, row in df.iterrows():
        data = row.to_dict()
        try:
            _ = MusicTrack(**data)
        except ValidationError as e:
            pass

for nrows in N_ROWS_TO_TEST:
    df_timing = pd.read_csv("dataset1.csv", nrows=nrows)
    df_timing.drop(columns=["Unnamed: 0"], inplace=True)
    df_timing = df_timing.dropna()
    text_cols = ["artists", "album_name", "track_name"]
    df_timing = df_timing[df_timing[text_cols].apply(lambda s: s.str.len().between(2, 512)).all(axis=1)]

    %timeit validation_pipe(df_timing)

104 µs ± 19.5 µs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
411 µs ± 21.1 µs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
9.2 ms ± 1.7 ms per loop (mean ± std. dev. of 7 runs, 100 loops each)
53.7 ms ± 3.85 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)
184 ms ± 51.8 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)
